In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Check what's in patches folder
patches_path = "/content/drive/MyDrive/mitosis_detector_BTTAI/BTTAI_Images/sanity_check"

print("Folders in patches/:")
print(os.listdir(patches_path))

# Check how many images in each folder
folder1_path = os.path.join(patches_path, "1")
folder2_path = os.path.join(patches_path, "2")

print(f"\nImages in folder '1': {len(os.listdir(folder1_path))}")
print(f"Images in folder '2': {len(os.listdir(folder2_path))}")

# Show first 5 files from each
print(f"\nFirst 5 files in folder '1':")
print(os.listdir(folder1_path)[:5])

print(f"\nFirst 5 files in folder '2':")
print(os.listdir(folder2_path)[:5])

Mounted at /content/drive
Folders in patches/:
['1', '2']

Images in folder '1': 64
Images in folder '2': 64

First 5 files in folder '1':
['2_2.jpeg', '2_1.jpeg', '2_4.jpeg', '2_5.jpeg', '2_12.jpeg']

First 5 files in folder '2':
['1_2.jpeg', '1_0.jpeg', '1_3.jpeg', '1_1.jpeg', '2_3.jpeg']


In [ ]:
import torch
import torch.nn as nn # neural network building block
import torch.optim as optim # optimizarion algorithm
from torch.utils.data import DataLoader, random_split # loads images in pabatches
from torchvision import datasets, transforms # image processing
from torchvision.models.segmentation import deeplabv3_resnet50
from tqdm import tqdm

In [ ]:
DATA_DIR = patches_path
IMG_SIZE = 224
BATCH_SIZE = 16 # processes 16 images at a time
NUM_EPOCHS = 20 # go through the entire dataset 20 rimes
LEARNING_RATE = 0.001

In [ ]:
# loads the deeplab model
"""
class DeepLabClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(DeepLabClassifier, self).__init__()
        self.deeplab = deeplabv3_resnet50(pretrained=pretrained)
        self.deeplab.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        seg_output = self.deeplab(x)['out']
        pooled = self.global_pool(seg_output)
        output = pooled.view(pooled.size(0), -1)
        return output
 """

"\nclass DeepLabClassifier(nn.Module):\n    def __init__(self, num_classes=2, pretrained=True):\n        super(DeepLabClassifier, self).__init__()\n        self.deeplab = deeplabv3_resnet50(pretrained=pretrained)\n        self.deeplab.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)\n        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))\n\n    def forward(self, x):\n        seg_output = self.deeplab(x)['out']\n        pooled = self.global_pool(seg_output)\n        output = pooled.view(pooled.size(0), -1)\n        return output\n "

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.hub.load('pytorch/vision:v0.10.0', 'deeplabv3_resnet50', pretrained=True)
model.classifier[4] = torch.nn.Conv2d(256, 2, kernel_size=1) ## changes classifier to 2 since it was 21 befor
model.aux_classifier[4] = torch.nn.Conv2d(256, 2, kernel_size=1) ### same thing to the auz classfier-> needed later
global_pool = torch.nn.AdaptiveAvgPool2d((1, 1))


Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /root/.cache/torch/hub/v0.10.0.zip


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:00<00:00, 202MB/s]


In [ ]:


# data augmentation
normalize = transforms.Normalize(
       mean=[0.485, 0.456, 0.406],  # Average colors from ImageNet
       std=[0.229, 0.224, 0.225]    # Standard deviations
   )
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    normalize,
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    normalize,
])

In [ ]:
full_dataset = datasets.ImageFolder(DATA_DIR)
#SUBSET_SIZE = 200
#full_dataset = torch.utils.data.Subset(full_dataset, range(min(SUBSET_SIZE, len(full_dataset))))
train_size = int(0.7 * len(full_dataset))  # trains 70 percent of the data
val_size = int(0.15 * len(full_dataset))   # trains 15 percent
test_size = len(full_dataset) - train_size - val_size  # test_size is 15 percent
# Create wrapper class to apply different transforms

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.subset)

# Randomly Split the dataset, into three sub
train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Wrap with transforms
train_dataset = TransformSubset(train_subset, train_transforms)
val_dataset = TransformSubset(val_subset, val_test_transforms)
test_dataset = TransformSubset(test_subset, val_test_transforms)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2) # loads data

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# lets see how accurate our model os
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in progress:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)['out']  # Extract 'out' from OrderedDict

        # Global pooling to convert segmentation output to classification
        outputs = global_pool(outputs)  # Shape: [batch, 2, 1, 1]
        outputs = outputs.squeeze(-1).squeeze(-1)  # Shape: [batch, 2]

        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        progress.set_postfix({'loss': loss.item(), 'acc': correct/total})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc




def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        progress = tqdm(dataloader, desc='Validating', leave=False)
        for images, labels in progress:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)['out']  # Extract 'out' from OrderedDict

            # Global pooling to convert segmentation output to classification
            outputs = global_pool(outputs)  # Shape: [batch, 2, 1, 1]
            outputs = outputs.squeeze(-1).squeeze(-1)  # Shape: [batch, 2]

            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    from sklearn.metrics import precision_score, recall_score, f1_score

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)

    return epoch_loss, epoch_acc, precision, recall, f1


In [ ]:
best_val_acc = 0.0
best_val_f1 = 0.0
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f" EPOCH {epoch+1}/{NUM_EPOCHS}")
    print('='*60)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc, val_precision, val_recall, val_f1 = validate(
        model, val_loader, criterion, device
    )

    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Print results
    print(f"\n RESULTS:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} ({val_acc*100:.2f}%)")
    print(f"   Precision:  {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")
    print(f"   Learning Rate: {current_lr:.6f}")

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)


 EPOCH 1/20


Validating:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



 RESULTS:
   Train Loss: 0.6344 | Train Acc: 0.6292 (62.92%)
   Val Loss:   294.8965 | Val Acc:   0.7895 (78.95%)
   Precision:  0.7895 | Recall: 1.0000 | F1: 0.8824
   Learning Rate: 0.001000

 EPOCH 2/20


Validating:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



 RESULTS:
   Train Loss: 0.6744 | Train Acc: 0.6404 (64.04%)
   Val Loss:   1518.6166 | Val Acc:   0.7895 (78.95%)
   Precision:  0.7895 | Recall: 1.0000 | F1: 0.8824
   Learning Rate: 0.001000

 EPOCH 3/20


...


 RESULTS:
   Train Loss: 0.2266 | Train Acc: 0.9101 (91.01%)
   Val Loss:   0.2926 | Val Acc:   0.8421 (84.21%)
   Precision:  1.0000 | Recall: 0.8000 | F1: 0.8889
   Learning Rate: 0.000250

 EPOCH 19/20


Validating:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



 RESULTS:
   Train Loss: 0.2640 | Train Acc: 0.8876 (88.76%)
   Val Loss:   0.8074 | Val Acc:   0.7895 (78.95%)
   Precision:  1.0000 | Recall: 0.7333 | F1: 0.8462
   Learning Rate: 0.000125

 EPOCH 20/20


Validating:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                                                         


 RESULTS:
   Train Loss: 0.2410 | Train Acc: 0.8876 (88.76%)
   Val Loss:   1.0707 | Val Acc:   0.5789 (57.89%)
   Precision:  1.0000 | Recall: 0.4667 | F1: 0.6364
   Learning Rate: 0.000125
